In [2]:
!pip install keras-tuner


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.4/129.4 kB 4.4 MB/s eta 0:00:00


In [54]:
from google.colab import files
uploaded = files.upload()


Saving WA_Fn-UseC_-Telco-Customer-Churn.csv to WA_Fn-UseC_-Telco-Customer-Churn (2).csv


In [55]:
df = pd.read_csv("WA_Fn-UseC_-Telco-Customer-Churn.csv")

In [68]:
print(df.isnull().sum())

customerID           0
gender               0
SeniorCitizen        0
Partner              0
Dependents           0
tenure               0
PhoneService         0
MultipleLines        0
InternetService      0
OnlineSecurity       0
OnlineBackup         0
DeviceProtection     0
TechSupport          0
StreamingTV          0
StreamingMovies      0
Contract             0
PaperlessBilling     0
PaymentMethod        0
MonthlyCharges       0
TotalCharges        11
Churn                0
dtype: int64


In [72]:
df['TotalCharges'].dropna(inplace=True)

In [75]:
df.isnull().sum()

,0
customerID,0
gender,0
SeniorCitizen,0
Partner,0
Dependents,0
tenure,0
PhoneService,0
MultipleLines,0
InternetService,0
OnlineSecurity,0


In [74]:
df = df.dropna()

In [77]:
print("NaN count:", np.isnan(X_scaled).sum())

print("Inf count:", np.isinf(X_scaled).sum())

X_scaled = np.nan_to_num(X_scaled)

NaN count: 7043
Inf count: 0


In [78]:
df = df.replace(" ", np.nan)
df = df.apply(pd.to_numeric, errors='coerce')
df = df.fillna(df.median())

In [88]:
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Dense, Dropout
from tensorflow.keras.regularizers import l2
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.preprocessing import MinMaxScaler, LabelEncoder
from sklearn.model_selection import train_test_split

# Load data
df = pd.read_csv("WA_Fn-UseC_-Telco-Customer-Churn.csv")
df.columns = df.columns.str.lower()
df = df.drop('customerid', axis=1)
df = df.replace(" ", np.nan)

# Numeric
num_cols = ['tenure','monthlycharges','totalcharges']
for col in num_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce')

# Binary
binary_cols = ['gender','partner','dependents','phoneservice','paperlessbilling']
for col in binary_cols:
    df[col] = df[col].map({'Yes':1,'No':0})

# Categorical
cat_cols = [
    'multiplelines','internetservice','onlinesecurity','onlinebackup',
    'deviceprotection','techsupport','streamingtv','streamingmovies',
    'contract','paymentmethod'
]
le = LabelEncoder()
for col in cat_cols:
    df[col] = le.fit_transform(df[col].astype(str))


# Features only
X = df.drop('churn', axis=1)
scaler = MinMaxScaler()
X_scaled = scaler.fit_transform(X)
X_scaled = np.nan_to_num(X_scaled)

X_train, X_test = train_test_split(X_scaled, test_size=0.2, random_state=42)

/usr/local/lib/python3.12/dist-packages/sklearn/utils/_array_api.py:776: RuntimeWarning: All-NaN slice encountered
  return xp.asarray(numpy.nanmin(X, axis=axis))
/usr/local/lib/python3.12/dist-packages/sklearn/utils/_array_api.py:793: RuntimeWarning: All-NaN slice encountered
  return xp.asarray(numpy.nanmax(X, axis=axis))


In [89]:
input_dim = X_scaled.shape[1]

input_layer = Input(shape=(input_dim,))
encoded = Dense(16, activation='relu')(input_layer)
decoded = Dense(input_dim, activation='sigmoid')(encoded)

baseline_ae = Model(inputs=input_layer, outputs=decoded)
baseline_ae.compile(optimizer='adam', loss='mse')

history_baseline = baseline_ae.fit(
    X_train, X_train,
    validation_split=0.2,
    epochs=20,
    batch_size=64,
    verbose=1
)

# Evaluate
X_pred_baseline = baseline_ae.predict(X_test)
mse_baseline = np.mean((X_test - X_pred_baseline)**2, axis=1)
threshold_baseline = np.mean(mse_baseline) + 2*np.std(mse_baseline)
anomalies_baseline = np.sum(mse_baseline > threshold_baseline)

print("Baseline MSE:", np.mean(mse_baseline))
print("Baseline anomalies:", anomalies_baseline)

Epoch 1/20
71/71 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - loss: 0.1883 - val_loss: 0.1759
Epoch 2/20
71/71 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.1602 - val_loss: 0.1372
Epoch 3/20
71/71 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.1203 - val_loss: 0.1012
Epoch 4/20
71/71 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.0918 - val_loss: 0.0786
Epoch 5/20
71/71 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.0730 - val_loss: 0.0632
Epoch 6/20
71/71 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.0595 - val_loss: 0.0520
Epoch 7/20
71/71 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.0493 - val_loss: 0.0436
Epoch 8/20
71/71 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.0414 - val_loss: 0.0368
Epoch 9/20
71/71 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.0350 - val_loss: 0.0313
Epoch 10/20
71/71 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.0296 - val_loss: 0.0266
Epoch 11/20
71/71 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.0252 - val_loss: 0.0227
Epoch 12/20
71/71 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.0214 - val_lo

In [93]:
# Add noise
noise_factor = 0.2
X_train_noisy = X_train + noise_factor*np.random.normal(loc=0.0, scale=1.0, size=X_train.shape)
X_train_noisy = np.clip(X_train_noisy, 0., 1.)
X_test_noisy = X_test + noise_factor*np.random.normal(loc=0.0, scale=1.0, size=X_test.shape)
X_test_noisy = np.clip(X_test_noisy, 0., 1.)

input_layer_adv = Input(shape=(input_dim,))
encoded_adv = Dense(32, activation='relu', kernel_regularizer=l2(0.001))(input_layer_adv)
encoded_adv = Dropout(0.2)(encoded_adv)
encoded_adv = Dense(16, activation='relu')(encoded_adv)

decoded_adv = Dense(32, activation='relu', kernel_regularizer=l2(0.001))(encoded_adv)
decoded_adv = Dense(input_dim, activation='sigmoid')(decoded_adv)

advanced_ae = Model(inputs=input_layer_adv, outputs=decoded_adv)
advanced_ae.compile(optimizer=Adam(learning_rate=0.001), loss='mse')

early_stop = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)

history_adv = advanced_ae.fit(
    X_train_noisy, X_train,
    validation_data=(X_test_noisy, X_test),
    epochs=20,
    batch_size=64,
    callbacks=[early_stop],
    verbose=1
)

# Evaluate
X_pred_adv = advanced_ae.predict(X_test_noisy)
mse_adv = np.mean((X_test - X_pred_adv)**2, axis=1)
threshold_adv = np.mean(mse_adv) + 2*np.std(mse_adv)
anomalies_adv = np.sum(mse_adv > threshold_adv)

print("Advanced MSE:", np.mean(mse_adv))
print("Advanced anomalies:", anomalies_adv)
print("Baseline MSE:", np.mean(mse_baseline))
print("Baseline anomalies:", anomalies_baseline)

Epoch 1/20
89/89 ━━━━━━━━━━━━━━━━━━━━ 2s 5ms/step - loss: 0.2068 - val_loss: 0.1727
Epoch 2/20
89/89 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.1527 - val_loss: 0.1295
Epoch 3/20
89/89 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.1228 - val_loss: 0.1070
Epoch 4/20
89/89 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.1076 - val_loss: 0.0947
Epoch 5/20
89/89 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 0.0994 - val_loss: 0.0884
Epoch 6/20
89/89 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.0945 - val_loss: 0.0845
Epoch 7/20
89/89 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.0911 - val_loss: 0.0810
Epoch 8/20
89/89 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.0881 - val_loss: 0.0781
Epoch 9/20
89/89 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.0858 - val_loss: 0.0756
Epoch 10/20
89/89 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.0838 - val_loss: 0.0737
Epoch 11/20
89/89 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.0821 - val_loss: 0.0723
Epoch 12/20
89/89 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.0803 - val_lo